In [1]:
import warnings
import time
warnings.filterwarnings('ignore')

import copy
import itertools

from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler

from load_dataset import *
from savage import *
from pipelines import *

%reload_ext autoreload
%autoreload 2

## Import and Split data

In [2]:
dataset = 'adult'
sens_attr = 'gender'

X_train, X_test, y_train, y_test = load(dataset)

# use sample for demonstrate efficiency
X_train = X_train.sample(frac=0.05, random_state=42)
X_test = X_test.sample(frac=0.1, random_state=42)
y_train = y_train.sample(frac=0.05, random_state=42)
y_test = y_test.sample(frac=0.1, random_state=42)

X_train_orig = copy.deepcopy(X_train).reset_index(drop=True)
X_test_orig = copy.deepcopy(X_test).reset_index(drop=True)
y_train, y_test = y_train.reset_index(drop=True), y_test.reset_index(drop=True)

## Run SAVAGE

**Load predefined pipeline.**

In [3]:
rf = RandomForestClassifier(n_estimators=30, random_state=42)
loaded_pipeline = make_pipeline_func('h2o', rf)

**Customized ML Pipeline: Impute with Iterative Imputer and Standardize, then train with non-differentiable random forest.**

In [4]:
def pipeline(X_train, y_train, X_test):
    imputer = IterativeImputer(random_state=42)
    model = RandomForestClassifier(n_estimators=30, random_state=42)
    X_train_imputed = imputer.fit_transform(X_train)
    ss = StandardScaler()
    ss.fit(X_train_imputed)
    model.fit(ss.transform(X_train_imputed), y_train)
    return model.predict_proba(ss.transform(X_test))

**Target Metric for Measuring Model Utility: AUC**

Design all metrics to be lower the worse

In [5]:
def auc(X_test, y_test, y_pred):
    return roc_auc_score(y_test, y_pred[:, 1])

**Target Metric for Measuring Model Unfairness: EOD**

In [6]:
# Equality of opportunity difference (EOD)
def eod(X_test, y_test, y_pred):
    # negative making sure lower the worse
    return -abs(np.mean(y_pred[X_test[((X_test[sens_attr] == 1) & (y_test == 1))].index][:, 1]) - \
                np.mean(y_pred[X_test[((X_test[sens_attr] == 0) & (y_test == 1))].index][:, 1]))

**Task: what is the worst-case harm caused by 10% of systematic missing data?**

In [7]:
# clean AUC
clean_auc = auc(X_test_orig, y_test, pipeline(X_train_orig, y_train, X_test_orig))
print(f'Clean AUC: {clean_auc}')

# clean EOD
clean_eod = eod(X_test_orig, y_test, pipeline(X_train_orig, y_train, X_test_orig))
print(f'Clean EOD: {clean_eod}')

Clean AUC: 0.8248880584260958
Clean EOD: -0.1785319152947205


In [8]:
# maximum number of missing data
budget_pct = 0.1
budget = int(X_train_orig.shape[0] * budget_pct)

In [10]:
# take top-3 patterns
top_k = 1

In [9]:
top_results = run_beam_search(X_train_orig, X_test_orig, y_train, y_test, pipeline, auc, budget, top_k=top_k)

Start Beam Search...


Beam search rounds:   0%|          | 0/2 [00:00<?, ?it/s]

Round 1 candidates:   0%|          | 0/1 [00:00<?, ?it/s]

Expanding candidate ['Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: age, id: 0, cols: ('age', 'Y')
Injected 150 errors (9.95%). Best value: 0.80716
target_col: workclass, id: 1, cols: ('workclass', 'Y')
Injected 125 errors (8.29%). Best value: 0.73582
target_col: education, id: 2, cols: ('education', 'Y')
Injected 150 errors (9.95%). Best value: 0.77379
target_col: marital, id: 3, cols: ('marital', 'Y')
Injected 150 errors (9.95%). Best value: 0.80652
target_col: relationship, id: 4, cols: ('relationship', 'Y')
Injected 150 errors (9.95%). Best value: 0.80820
target_col: race, id: 5, cols: ('race', 'Y')
Injected 28 errors (1.86%). Best value: 0.80697
target_col: gender, id: 6, cols: ('gender', 'Y')
Injected 150 errors (9.95%). Best value: 0.81206
target_col: hours, id: 7, cols: ('hours', 'Y')
Injected 150 errors (9.95%). Best value: 0.80263
----------- ROUND BEST -----------
[(['workclass', 'Y'], 0.7358201642949106), (['education', 'Y'], 0.7737914297758889), (['hours', 'Y'], 0.8026250726338456)]
----------------------------------


...

target_col: workclass, id: 1, cols: ('workclass', 'age', 'Y')
Injected 143 errors (9.48%). Best value: 0.73894
target_col: workclass, id: 1, cols: ('workclass', 'education', 'Y')
Injected 147 errors (9.75%). Best value: 0.72823
target_col: workclass, id: 1, cols: ('workclass', 'marital', 'Y')
Injected 76 errors (5.04%). Best value: 0.78649
target_col: workclass, id: 1, cols: ('workclass', 'relationship', 'Y')
Injected 150 errors (9.95%). Best value: 0.80901
target_col: workclass, id: 1, cols: ('workclass', 'race', 'Y')
Injected 125 errors (8.29%). Best value: 0.73582
target_col: workclass, id: 1, cols: ('workclass', 'gender', 'Y')
Injected 150 errors (9.95%). Best value: 0.80901
target_col: workclass, id: 1, cols: ('workclass', 'hours', 'Y')
Injected 150 errors (9.95%). Best value: 0.73012


Expanding candidate ['education', 'Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: education, id: 2, cols: ('education', 'age', 'Y')
Injected 150 errors (9.95%). Best value: 0.73950
target_col: education, id: 2, cols: ('education', 'workclass', 'Y')
Injected 87 errors (5.77%). Best value: 0.77595
target_col: education, id: 2, cols: ('education', 'marital', 'Y')
Injected 150 errors (9.95%). Best value: 0.79470
target_col: education, id: 2, cols: ('education', 'relationship', 'Y')
Injected 84 errors (5.57%). Best value: 0.76533
target_col: education, id: 2, cols: ('education', 'race', 'Y')
Injected 87 errors (5.77%). Best value: 0.77595
target_col: education, id: 2, cols: ('education', 'gender', 'Y')
Injected 84 errors (5.57%). Best value: 0.76533
target_col: education, id: 2, cols: ('education', 'hours', 'Y')
Injected 84 errors (5.57%). Best value: 0.76533


Expanding candidate ['hours', 'Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: hours, id: 7, cols: ('hours', 'age', 'Y')
Injected 150 errors (9.95%). Best value: 0.76325
target_col: hours, id: 7, cols: ('hours', 'workclass', 'Y')
Injected 118 errors (7.82%). Best value: 0.76924
target_col: hours, id: 7, cols: ('hours', 'education', 'Y')
Injected 150 errors (9.95%). Best value: 0.73168
target_col: hours, id: 7, cols: ('hours', 'marital', 'Y')
Injected 150 errors (9.95%). Best value: 0.73345
target_col: hours, id: 7, cols: ('hours', 'relationship', 'Y')
Injected 150 errors (9.95%). Best value: 0.68493
target_col: hours, id: 7, cols: ('hours', 'race', 'Y')
Injected 150 errors (9.95%). Best value: 0.77500
target_col: hours, id: 7, cols: ('hours', 'gender', 'Y')
Injected 150 errors (9.95%). Best value: 0.77481
----------- ROUND BEST -----------
[(['hours', 'relationship', 'Y'], 0.6849320375075483), (['workclass', 'education', 'Y'], 0.7282320637127004), (['workclass', 'hours', 'Y'], 0.7301222527316023)]
----------------------------------
Beam Search executi

In [10]:
r = top_results[0]
print(f'Missing data in column {r[0][0]} depending on columns {r[0]} could lead to an AUC drop of {clean_auc-r[1]}')

Missing data in column hours depending on columns ('hours', 'relationship', 'Y') could lead to an AUC drop of 0.13995602091854753


In [11]:
top_results = run_beam_search(X_train_orig, X_test_orig, y_train, y_test, pipeline, eod, budget, top_k=top_k)

Start Beam Search...


Beam search rounds:   0%|          | 0/2 [00:00<?, ?it/s]

Round 1 candidates:   0%|          | 0/1 [00:00<?, ?it/s]

Expanding candidate ['Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: age, id: 0, cols: ('age', 'Y')
Injected 150 errors (9.95%). Best value: -0.19971
target_col: workclass, id: 1, cols: ('workclass', 'Y')
Injected 85 errors (5.64%). Best value: -0.19684
target_col: education, id: 2, cols: ('education', 'Y')
Injected 150 errors (9.95%). Best value: -0.20404
target_col: marital, id: 3, cols: ('marital', 'Y')
Injected 56 errors (3.71%). Best value: -0.19472
target_col: relationship, id: 4, cols: ('relationship', 'Y')
Injected 60 errors (3.98%). Best value: -0.19861
target_col: race, id: 5, cols: ('race', 'Y')
Injected 150 errors (9.95%). Best value: -0.19517
target_col: gender, id: 6, cols: ('gender', 'Y')
Injected 54 errors (3.58%). Best value: -0.46344
target_col: hours, id: 7, cols: ('hours', 'Y')
Injected 150 errors (9.95%). Best value: -0.19974
----------- ROUND BEST -----------
[(['gender', 'Y'], -0.46344409240728646), (['education', 'Y'], -0.20403640567082082), (['hours', 'Y'], -0.1997444263452968)]
----------------------------------


...

target_col: gender, id: 6, cols: ('gender', 'age', 'Y')
Injected 54 errors (3.58%). Best value: -0.46344
target_col: gender, id: 6, cols: ('gender', 'workclass', 'Y')
Injected 33 errors (2.19%). Best value: -0.34731
target_col: gender, id: 6, cols: ('gender', 'education', 'Y')
Injected 54 errors (3.58%). Best value: -0.46344
target_col: gender, id: 6, cols: ('gender', 'marital', 'Y')
Injected 54 errors (3.58%). Best value: -0.46344
target_col: gender, id: 6, cols: ('gender', 'relationship', 'Y')
Injected 54 errors (3.58%). Best value: -0.46344
target_col: gender, id: 6, cols: ('gender', 'race', 'Y')
Injected 54 errors (3.58%). Best value: -0.46344
target_col: gender, id: 6, cols: ('gender', 'hours', 'Y')
Injected 54 errors (3.58%). Best value: -0.46344


Expanding candidate ['education', 'Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: education, id: 2, cols: ('education', 'age', 'Y')
Injected 150 errors (9.95%). Best value: -0.21142
target_col: education, id: 2, cols: ('education', 'workclass', 'Y')
Injected 49 errors (3.25%). Best value: -0.19761
target_col: education, id: 2, cols: ('education', 'marital', 'Y')
Injected 53 errors (3.51%). Best value: -0.20941
target_col: education, id: 2, cols: ('education', 'relationship', 'Y')
Injected 147 errors (9.75%). Best value: -0.26487
target_col: education, id: 2, cols: ('education', 'race', 'Y')
Injected 150 errors (9.95%). Best value: -0.19694
target_col: education, id: 2, cols: ('education', 'gender', 'Y')
Injected 129 errors (8.55%). Best value: -0.23457
target_col: education, id: 2, cols: ('education', 'hours', 'Y')
Injected 150 errors (9.95%). Best value: -0.24331


Expanding candidate ['hours', 'Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: hours, id: 7, cols: ('hours', 'age', 'Y')
Injected 100 errors (6.63%). Best value: -0.22547
target_col: hours, id: 7, cols: ('hours', 'workclass', 'Y')
Injected 138 errors (9.15%). Best value: -0.20477
target_col: hours, id: 7, cols: ('hours', 'education', 'Y')
Injected 150 errors (9.95%). Best value: -0.20054
target_col: hours, id: 7, cols: ('hours', 'marital', 'Y')
Injected 139 errors (9.22%). Best value: -0.28545
target_col: hours, id: 7, cols: ('hours', 'relationship', 'Y')
Injected 150 errors (9.95%). Best value: -0.20330
target_col: hours, id: 7, cols: ('hours', 'race', 'Y')
Injected 150 errors (9.95%). Best value: -0.19974
target_col: hours, id: 7, cols: ('hours', 'gender', 'Y')
Injected 54 errors (3.58%). Best value: -0.39979
----------- ROUND BEST -----------
[(['gender', 'age', 'Y'], -0.46344409240728646), (['gender', 'education', 'Y'], -0.46344409240728646), (['gender', 'marital', 'Y'], -0.46344409240728646)]
----------------------------------
Beam Search executi

In [12]:
r = top_results[0]
print(f'Missing data in column {r[0][0]} depending on columns {r[0]} could lead to an unfairness increase of {clean_eod-r[1]}')
    

Missing data in column gender depending on columns ('gender', 'Y') could lead to an unfairness increase of 0.28491217711256595


**Evaluation for loaded pipeline**

In [11]:
top_results = run_beam_search(X_train_orig, X_test_orig, y_train, y_test, loaded_pipeline, auc, budget, top_k=top_k)

Start Beam Search...


Beam search rounds:   0%|          | 0/2 [00:00<?, ?it/s]

Round 1 candidates:   0%|          | 0/1 [00:00<?, ?it/s]

Expanding candidate ['Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: age, id: 0, cols: ('age', 'Y')
Injected 150 errors (9.95%). Best value: 0.80716
target_col: workclass, id: 1, cols: ('workclass', 'Y')
Injected 109 errors (7.23%). Best value: 0.75343
target_col: education, id: 2, cols: ('education', 'Y')
Injected 150 errors (9.95%). Best value: 0.76626
target_col: marital, id: 3, cols: ('marital', 'Y')
Injected 150 errors (9.95%). Best value: 0.80883
target_col: relationship, id: 4, cols: ('relationship', 'Y')
Injected 150 errors (9.95%). Best value: 0.80735
target_col: race, id: 5, cols: ('race', 'Y')
Injected 28 errors (1.86%). Best value: 0.80663
target_col: gender, id: 6, cols: ('gender', 'Y')
Injected 150 errors (9.95%). Best value: 0.81241
target_col: hours, id: 7, cols: ('hours', 'Y')
Injected 150 errors (9.95%). Best value: 0.80263
----------- ROUND BEST -----------
[(['workclass', 'Y'], 0.7534277478380749), (['education', 'Y'], 0.766258018206884), (['hours', 'Y'], 0.8026250726338456)]
----------------------------------


...

target_col: workclass, id: 1, cols: ('workclass', 'age', 'Y')
Injected 143 errors (9.48%). Best value: 0.73349
target_col: workclass, id: 1, cols: ('workclass', 'education', 'Y')
Injected 147 errors (9.75%). Best value: 0.73582
target_col: workclass, id: 1, cols: ('workclass', 'marital', 'Y')
Injected 84 errors (5.57%). Best value: 0.77963
target_col: workclass, id: 1, cols: ('workclass', 'relationship', 'Y')
Injected 125 errors (8.29%). Best value: 0.72810
target_col: workclass, id: 1, cols: ('workclass', 'race', 'Y')
Injected 125 errors (8.29%). Best value: 0.72810
target_col: workclass, id: 1, cols: ('workclass', 'gender', 'Y')
Injected 150 errors (9.95%). Best value: 0.80474
target_col: workclass, id: 1, cols: ('workclass', 'hours', 'Y')
Injected 139 errors (9.22%). Best value: 0.72703


Expanding candidate ['education', 'Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: education, id: 2, cols: ('education', 'age', 'Y')
Injected 118 errors (7.82%). Best value: 0.73701
target_col: education, id: 2, cols: ('education', 'workclass', 'Y')
Injected 70 errors (4.64%). Best value: 0.78648
target_col: education, id: 2, cols: ('education', 'marital', 'Y')
Injected 45 errors (2.98%). Best value: 0.80133
target_col: education, id: 2, cols: ('education', 'relationship', 'Y')
Injected 150 errors (9.95%). Best value: 0.77698
target_col: education, id: 2, cols: ('education', 'race', 'Y')
Injected 57 errors (3.78%). Best value: 0.79954
target_col: education, id: 2, cols: ('education', 'gender', 'Y')
Injected 84 errors (5.57%). Best value: 0.76932
target_col: education, id: 2, cols: ('education', 'hours', 'Y')
Injected 84 errors (5.57%). Best value: 0.76932


Expanding candidate ['hours', 'Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: hours, id: 7, cols: ('hours', 'age', 'Y')
Injected 150 errors (9.95%). Best value: 0.76325
target_col: hours, id: 7, cols: ('hours', 'workclass', 'Y')
Injected 118 errors (7.82%). Best value: 0.76924
target_col: hours, id: 7, cols: ('hours', 'education', 'Y')
Injected 150 errors (9.95%). Best value: 0.73168
target_col: hours, id: 7, cols: ('hours', 'marital', 'Y')
Injected 150 errors (9.95%). Best value: 0.73345
target_col: hours, id: 7, cols: ('hours', 'relationship', 'Y')
Injected 150 errors (9.95%). Best value: 0.68493
target_col: hours, id: 7, cols: ('hours', 'race', 'Y')
Injected 150 errors (9.95%). Best value: 0.77500
target_col: hours, id: 7, cols: ('hours', 'gender', 'Y')
Injected 150 errors (9.95%). Best value: 0.77481
----------- ROUND BEST -----------
[(['hours', 'relationship', 'Y'], 0.6849320375075483), (['workclass', 'hours', 'Y'], 0.727027766067746), (['workclass', 'relationship', 'Y'], 0.7281044560152218)]
----------------------------------
Beam Search execu

In [12]:
r = top_results[0]
print(f'Missing data in column {r[0][0]} depending on columns {r[0]} could lead to an AUC drop of {clean_auc-r[1]}')

Missing data in column hours depending on columns ('hours', 'relationship', 'Y') could lead to an AUC drop of 0.13995602091854753
